In [ ]:
# scripts/10_compute_distance_matrix.py

"""
Step 10 - Compute the functional distance matrix (extended metric).

Inputs:
- outputs/data/functional_space.parquet
- outputs/models/functional_knn_params.pkl

Outputs:
- outputs/matrices/functional_distance_matrix.npy
- outputs/matrices/functional_distance_matrix_nits.npy
- outputs/reports/step10_distance_matrix_summary.txt
"""

from __future__ import annotations

import pickle
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm import tqdm


# ---------------------------------------------------------------------
# Step 1: Configuration
# ---------------------------------------------------------------------
SPACEF_PATH = Path("outputs") / "data" / "functional_space.parquet"
PARAMS_PATH = Path("outputs") / "models" / "functional_knn_params.pkl"

DIST_MATRIX_PATH = Path("outputs") / "matrices" / "functional_distance_matrix.npy"
NITS_ORDER_PATH = Path("outputs") / "matrices" / "functional_distance_matrix_nits.npy"
SUMMARY_PATH = Path("outputs") / "reports" / "step10_distance_matrix_summary.txt"

# If your data span is 25 years (YEAR_MAX-24..YEAR_MAX), max gap ~ 24
DEFAULT_MAX_YEAR_GAP = 24.0

# Ensure output folders exist
for p in [DIST_MATRIX_PATH, NITS_ORDER_PATH, SUMMARY_PATH]:
    p.parent.mkdir(parents=True, exist_ok=True)


# ---------------------------------------------------------------------
# Step 2: Load inputs
# ---------------------------------------------------------------------
spaceF = pd.read_parquet(SPACEF_PATH)
print(f"[OK] Functional space loaded: {spaceF.shape}")

with open(PARAMS_PATH, "rb") as f:
    params = pickle.load(f)

k = int(params.get("k"))
lambda_p = float(params.get("lambda"))

# Financial weights: accept either "weights" (repo) or legacy "pesos"
fin_weights = params.get("weights", None)
if fin_weights is None:
    fin_weights = params.get("pesos", None)
if fin_weights is None:
    raise KeyError("Could not find 'weights' (or legacy 'pesos') inside the params file.")

# Extra weights (repo-friendly names first)
w_dep = params.get("weight_DEP", None)
w_ciiu = params.get("weight_CIIU", None)
w_year = params.get("weight_year", None)
max_year_gap = float(params.get("max_year_gap", DEFAULT_MAX_YEAR_GAP))

# Fallback to legacy 20_ naming if needed
# (Note: in your 20_ code you used 'peso_DEP', 'peso_CIIU', 'peso_desfase')
if w_dep is None:
    w_dep = params.get("peso_DEP", None)
if w_ciiu is None:
    w_ciiu = params.get("peso_CIIU", None)
if w_year is None:
    w_year = params.get("peso_desfase", None)

# If still missing, default to 0.0 (meaning: pure functional distance)
w_dep = float(w_dep) if w_dep is not None else 0.0
w_ciiu = float(w_ciiu) if w_ciiu is not None else 0.0
w_year = float(w_year) if w_year is not None else 0.0

# Validate required extra columns if extra weights are used
required_cols = []
if w_dep > 0:
    required_cols.append("DEP")
if w_ciiu > 0:
    required_cols.append("CIIU_Letra")
if w_year > 0:
    required_cols.append("Año_final")

missing_cols = [c for c in required_cols if c not in spaceF.columns]
if missing_cols:
    raise KeyError(
        f"Step 10 requires columns {missing_cols} in functional_space.parquet "
        f"because the params file includes positive extra weights."
    )


# ---------------------------------------------------------------------
# Step 3: Identify functional columns and indicators
# ---------------------------------------------------------------------
functional_cols = [c for c in spaceF.columns if "_-" in c]
indicators = sorted({c.split("_")[0] for c in functional_cols})

# Window size inferred from suffixes: e.g. var_-0 ... var_-4 -> n_window = 5
suffixes = {c.split("_")[1] for c in functional_cols if len(c.split("_")) >= 2}
n_window = len(suffixes)

print(
    f"[INFO] Indicators: {len(indicators)} | Window length: {n_window} | "
    f"k: {k} | lambda: {lambda_p:.4f} | "
    f"w_dep: {w_dep:.4f} | w_ciiu: {w_ciiu:.4f} | w_year: {w_year:.4f} | "
    f"max_year_gap: {max_year_gap:.1f}"
)


# ---------------------------------------------------------------------
# Step 4: Extended weighted distance
# ---------------------------------------------------------------------
def extended_weighted_distance(
    f1: pd.Series,
    f2: pd.Series,
    lambda_p: float,
    n: int,
    fin_weights: dict,
    w_dep: float,
    w_ciiu: float,
    w_year: float,
    max_year_gap: float,
) -> float:
    """
    Distance = weighted average of:
      - functional bounded distances per financial indicator (with missingness penalty)
      - + categorical DEP mismatch (0/1)
      - + categorical CIIU mismatch (0/1)
      - + normalized year gap in [0,1] = |year1-year2|/max_year_gap
    """

    total, wsum = 0.0, 0.0

    # --- Part 1: financial functional distance (bounded) ---
    for var in indicators:
        v1 = [f1.get(f"{var}_-{i}", np.nan) for i in range(n)]
        v2 = [f2.get(f"{var}_-{i}", np.nan) for i in range(n)]

        l1, valid = 0.0, 0
        for a, b in zip(v1, v2):
            if pd.notna(a) and pd.notna(b):
                if np.isinf(a) and np.isinf(b) and a == b:
                    l1 += 0.0
                elif np.isinf(a) or np.isinf(b):
                    l1 += np.inf
                else:
                    l1 += abs(a - b)
                valid += 1

        missing = n - valid
        if valid > 0:
            penalized = l1 * (1 + lambda_p * (missing / n))
            bounded = penalized / (1 + penalized) if np.isfinite(penalized) else 1.0
        else:
            bounded = 1.0

        w = float(fin_weights.get(var, 1.0))
        total += w * bounded
        wsum += w

    # --- Part 2: DEP (0/1) ---
    if w_dep > 0.0:
        d_dep = 0.0 if f1["DEP"] == f2["DEP"] else 1.0
        total += w_dep * d_dep
        wsum += w_dep

    # --- Part 3: CIIU (0/1) ---
    if w_ciiu > 0.0:
        d_ciiu = 0.0 if f1["CIIU_Letra"] == f2["CIIU_Letra"] else 1.0
        total += w_ciiu * d_ciiu
        wsum += w_ciiu

    # --- Part 4: year gap normalized to [0,1] ---
    if w_year > 0.0:
        gap = abs(float(f1["Año_final"]) - float(f2["Año_final"]))
        denom = max(float(max_year_gap), 1.0)
        d_year = min(gap / denom, 1.0)
        total += w_year * d_year
        wsum += w_year

    return total / wsum if wsum > 0 else 1.0


# ---------------------------------------------------------------------
# Step 5: Compute distance matrix
# ---------------------------------------------------------------------
n_firms = spaceF.shape[0]
nits_order = spaceF.index.tolist()

dist_matrix = np.zeros((n_firms, n_firms), dtype=np.float32)

print("[RUN] Computing extended functional distance matrix...")
for i in tqdm(range(n_firms), desc="Pairwise distances"):
    f1 = spaceF.iloc[i]
    for j in range(i + 1, n_firms):
        f2 = spaceF.iloc[j]
        d = extended_weighted_distance(
            f1=f1,
            f2=f2,
            lambda_p=lambda_p,
            n=n_window,
            fin_weights=fin_weights,
            w_dep=w_dep,
            w_ciiu=w_ciiu,
            w_year=w_year,
            max_year_gap=max_year_gap,
        )
        dist_matrix[i, j] = d
        dist_matrix[j, i] = d  # symmetry


# ---------------------------------------------------------------------
# Step 6: Save outputs
# ---------------------------------------------------------------------
np.save(DIST_MATRIX_PATH, dist_matrix)
np.save(NITS_ORDER_PATH, np.array(nits_order, dtype=object))

print(f"[SAVED] Distance matrix: {DIST_MATRIX_PATH}")
print(f"[SAVED] NIT order:       {NITS_ORDER_PATH}")


# ---------------------------------------------------------------------
# Step 7: Summary report
# ---------------------------------------------------------------------
with open(SUMMARY_PATH, "w", encoding="utf-8") as f:
    f.write("Step 10 - Functional Distance Matrix Summary (Extended Metric)\n")
    f.write("============================================================\n\n")

    f.write("Inputs\n")
    f.write(f" - Functional space: {SPACEF_PATH.as_posix()}\n")
    f.write(f" - Params file:      {PARAMS_PATH.as_posix()}\n\n")

    f.write("Metric configuration\n")
    f.write(f" - k: {k}\n")
    f.write(f" - lambda: {lambda_p:.4f}\n")
    f.write(f" - Indicators: {len(indicators)}\n")
    f.write(f" - Window length: {n_window}\n")
    f.write(f" - Extra weights: DEP={w_dep:.4f}, CIIU={w_ciiu:.4f}, year={w_year:.4f}\n")
    f.write(f" - Year gap normalization (max_year_gap): {max_year_gap:.1f}\n\n")

    f.write("Outputs\n")
    f.write(f" - Distance matrix: {DIST_MATRIX_PATH.as_posix()}\n")
    f.write(f" - NIT order:       {NITS_ORDER_PATH.as_posix()}\n\n")

    f.write("Matrix stats\n")
    f.write(f" - Firms: {n_firms:,}\n")
    f.write(f" - Shape: {dist_matrix.shape}\n")

print(f"[SAVED] Summary: {SUMMARY_PATH}")